In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pilot')
    .master('local[*]')
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 15:06:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/14 15:06:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
emp_df = spark.read.csv('/home/spark/data/employees_tax', header=True)

In [7]:
emp_df.show()

+-----------+-------------+-------------+---+------+-------+----------+-------+---+---+
|employee_id|department_id|         name|age|gender| salary| hire_date|    tax|one|two|
+-----------+-------------+-------------+---+------+-------+----------+-------+---+---+
|        015|          106|  Michael Lee| 37|  Male|63000.0|2014-09-30|12600.0|  1|two|
|        016|          107|  Kelly Zhang| 30|Female|49000.0|2018-04-01| 9800.0|  1|two|
|        007|          101|James Johnson| 42|  Male|70000.0|2012-03-15|14000.0|  1|two|
|        008|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|10200.0|  1|two|
|        011|          104|   David Park| 38|  Male|65000.0|2015-11-01|13000.0|  1|two|
|        012|          105|   Susan Chen| 31|Female|54000.0|2017-02-15|10800.0|  1|two|
|        019|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|12400.0|  1|two|
|        020|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|10600.0|  1|two|
|        003|          102|    B

In [10]:
emp_df.select('department_id').distinct().show()

+-------------+
|department_id|
+-------------+
|          107|
|          106|
|          101|
|          102|
|          104|
|          105|
|          103|
+-------------+



In [19]:
# Window function, max salary for department

from pyspark.sql.window import Window
from pyspark.sql.functions import col, max, desc

window = Window.partitionBy('department_id').orderBy(col('salary').desc())
max_func = max('salary').over(window)

emp_df.withColumn('max_dpt_salary', max_func).show()

+-----------+-------------+-------------+---+------+-------+----------+-------+---+---+--------------+
|employee_id|department_id|         name|age|gender| salary| hire_date|    tax|one|two|max_dpt_salary|
+-----------+-------------+-------------+---+------+-------+----------+-------+---+---+--------------+
|        007|          101|James Johnson| 42|  Male|70000.0|2012-03-15|14000.0|  1|two|       70000.0|
|        001|          101|     John Doe| 30|  Male|50000.0|2015-01-01|10000.0|  1|two|       70000.0|
|        002|          101|   Jane Smith| 25|Female|45000.0|2016-02-15| 9000.0|  1|two|       70000.0|
|        003|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|11000.0|  1|two|       55000.0|
|        020|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|10600.0|  1|two|       55000.0|
|        008|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|10200.0|  1|two|       55000.0|
|        004|          102|    Alice Lee| 28|Female|48000.0|2017-09-30| 9

In [20]:
emp_df.groupBy('department_id').agg(max(col('salary')).alias('salary')).show()

+-------------+-------+
|department_id| salary|
+-------------+-------+
|          101|70000.0|
|          102|55000.0|
|          103|62000.0|
|          104|65000.0|
|          105|57000.0|
|          106|75000.0|
|          107|49000.0|
+-------------+-------+



In [23]:
# row_number(), department max salary rank

from pyspark.sql.window import Window
from pyspark.sql.functions import col, desc, row_number

window = Window.partitionBy('department_id').orderBy(col('salary').desc())
row_number_func = row_number().over(window)

emp_df.withColumn('dprt_rank', row_number_func).show()

+-----------+-------------+-------------+---+------+-------+----------+-------+---+---+---------+
|employee_id|department_id|         name|age|gender| salary| hire_date|    tax|one|two|dprt_rank|
+-----------+-------------+-------------+---+------+-------+----------+-------+---+---+---------+
|        007|          101|James Johnson| 42|  Male|70000.0|2012-03-15|14000.0|  1|two|        1|
|        001|          101|     John Doe| 30|  Male|50000.0|2015-01-01|10000.0|  1|two|        2|
|        002|          101|   Jane Smith| 25|Female|45000.0|2016-02-15| 9000.0|  1|two|        3|
|        003|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|11000.0|  1|two|        1|
|        020|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|10600.0|  1|two|        2|
|        008|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|10200.0|  1|two|        3|
|        004|          102|    Alice Lee| 28|Female|48000.0|2017-09-30| 9600.0|  1|two|        4|
|        019|       

In [29]:
from pyspark.sql.functions import expr

emp_df.withColumn('dprt_rank', expr('row_number() over(partition by department_id order by salary desc)')).where('dprt_rank = 2').show()

+-----------+-------------+-------------+---+------+-------+----------+-------+---+---+---------+
|employee_id|department_id|         name|age|gender| salary| hire_date|    tax|one|two|dprt_rank|
+-----------+-------------+-------------+---+------+-------+----------+-------+---+---+---------+
|        007|          101|James Johnson| 42|  Male|70000.0|2012-03-15|14000.0|  1|two|        1|
|        001|          101|     John Doe| 30|  Male|50000.0|2015-01-01|10000.0|  1|two|        2|
|        002|          101|   Jane Smith| 25|Female|45000.0|2016-02-15| 9000.0|  1|two|        3|
|        003|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|11000.0|  1|two|        1|
|        020|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|10600.0|  1|two|        2|
|        008|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|10200.0|  1|two|        3|
|        004|          102|    Alice Lee| 28|Female|48000.0|2017-09-30| 9600.0|  1|two|        4|
|        019|       